# Neo4j agent memory for Haystack

Three kinds of memory for a Haystack chatbot or agent, stored with [`neo4j-agent-memory`](https://neo4j.com/labs/agent-memory/):

| Memory | What is kept |
| --- | --- |
| **Short-term** | The conversation of each session |
| **Long-term** | Preferences, facts and entities that outlive sessions |
| **Reasoning** | Traces of past tasks: tool calls, outcome, success or failure |

This notebook shows four ways to use it:

1. A **pipeline** that reads memory before the LLM and saves all three kinds after it
2. **Tools and reasoning memory** with an `Agent` inside the pipeline
3. **Saving memory directly** through the store, without a pipeline
4. **Memory tools** (`retrieve_memories`, `store_memory`) that an `Agent` calls when it decides to

It runs on either backend. Set `BACKEND` below:

- `"bolt"`: your own Neo4j database (local Docker, Aura, …). All memory types.
- `"nams"`: the hosted Neo4j Agent Memory Service. Memory shows up in your NAMS workspace. NAMS extracts entities itself and does not store preferences or facts; those steps are skipped with a warning.

In [ ]:
# From a clone of the repo (or: pip install "haystack-neo4j[memory] @ git+https://github.com/neo4j-labs/neo4j-haystack")
%pip install -q -e "..[memory]" python-dotenv

## Credentials

Keep them in a `.env` file next to the repo, never in the notebook:

```
OPENAI_API_KEY=...
# bolt
NEO4J_URI=neo4j+s://<id>.databases.neo4j.io
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=...
NEO4J_DATABASE=neo4j
# nams
NAMS_API_KEY=...
NAMS_WORKSPACE_ID=...
```

Don't point `bolt` at the database behind a NAMS workspace: the two use different schemas.

In [ ]:
import os
from getpass import getpass

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

BACKEND = "bolt"  # or "nams"
print("Backend:", BACKEND, "| OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## 1. Configure the store

`Neo4jMemoryStore` takes the library's own `MemorySettings`, like `MemoryClient` does. Secrets stay out of the settings: the store reads them from `NEO4J_PASSWORD` / `NAMS_API_KEY` (or `password=` / `nams_api_key=` `Secret`s), so saved pipelines contain variable names only.

In [ ]:
from neo4j_agent_memory import ExtractionConfig, MemorySettings, NamsConfig

from haystack_integrations.memory_stores.neo4j import Neo4jMemoryStore

if BACKEND == "bolt":
    settings = MemorySettings(
        # Connection: NEO4J_URI, NEO4J_USERNAME, NEO4J_DATABASE from .env, or e.g.
        # neo4j={"uri": "neo4j+s://...", "database": "neo4j"}  (as a dict, without the password)
        embedding="openai/text-embedding-3-small",
        extraction=ExtractionConfig(extractor_type="llm", entity_types=["PERSON", "ORGANIZATION", "LOCATION"]),
    )
else:
    settings = MemorySettings(backend="nams", nams=NamsConfig(workspace_id=os.environ["NAMS_WORKSPACE_ID"]))

store = Neo4jMemoryStore(settings)
print("Backend:", store.backend, "| settings:", store.settings)

## 2. A pipeline that saves all three kinds of memory

`Neo4jMemoryRetriever` reads memory before the LLM; `Neo4jMemoryWriter` saves the turn after it:

- **short-term**: the question and the answer, in the session
- **long-term**: entities extracted under the `ExtractionPolicy` (here `review`: only confident ones), plus preferences and facts you pass in
- **reasoning**: a trace of the turn

Sessions come from `store.create_session()`: NAMS assigns conversation ids, and on bolt it returns a new id.

In [ ]:
from haystack import Pipeline
from haystack.components.generators.chat import OpenAIChatGenerator

from haystack_integrations.components.retrievers.neo4j import Neo4jMemoryRetriever
from haystack_integrations.components.writers.neo4j import Neo4jMemoryWriter
from haystack_integrations.memory_stores.neo4j import ExtractionPolicy

policy = ExtractionPolicy(user="review", assistant="review", min_confidence=0.7)

pipe = Pipeline()
pipe.add_component("reader", Neo4jMemoryRetriever(store, system_prompt="You are a concise research assistant."))
pipe.add_component("llm", OpenAIChatGenerator(model="gpt-5.4-mini"))
pipe.add_component("writer", Neo4jMemoryWriter(store, extraction_policy=policy))
pipe.connect("reader.messages", "llm.messages")
pipe.connect("llm.replies", "writer.replies")


def chat(query, session_id, **writer_inputs):
    result = pipe.run(
        {
            "reader": {"query": query, "session_id": session_id},
            "writer": {"query": query, "session_id": session_id, **writer_inputs},
        },
        include_outputs_from={"reader", "llm"},  # these feed other components, so ask for them
    )
    print("Memory the model saw:\n", result["reader"]["context"] or "(nothing recalled yet)")
    print("\nAnswer:", result["llm"]["replies"][0].text)
    print("\nSaved:", {k: result["writer"][k] for k in ("messages_written", "trace_id")})
    print("Entities linked:", [(e["name"], e["type"], e["role"]) for e in result["writer"]["entities"]])
    return result


session_1 = store.create_session(user_id="pravesh")
print("Session 1 ID:", session_1)  # create_session returns the id as a string

### Turn 1: a question, with a preference and a fact to remember

In [ ]:
first = chat(
    "I'm Pravesh from Northwind Capital. I focus on German AI companies. Which ones should I look at?",
    session_1,
    user_id="pravesh",
    preferences=[{"category": "region", "preference": "Focus on companies in Germany"}],
    facts=[{"subject": "Pravesh", "predicate": "works_at", "object": "Northwind Capital"}],
)

### Turn 2, same session: short-term memory resolves "those"

In [ ]:
second = chat("Which of those has the largest team?", session_1, user_id="pravesh")
print("\nHistory replayed:", [m.text[:50] for m in second["reader"]["history"]])

### Turn 3, new session: long-term and reasoning memory carry over

In [ ]:
session_2 = store.create_session(user_id="pravesh")
print("Session 2 ID:", session_2)
third = chat("Suggest a few companies for my next deep dive.", session_2, user_id="pravesh")
memories = third["reader"]["memories"]
print("\nPreferences:", [p["preference"] for p in memories["preferences"]])
print("Entities:", [e["name"] for e in memories["entities"]])
print("Facts:", [f"{f['subject']} {f['predicate']} {f['object']}" for f in memories["facts"]])
print("Past tasks:", [t["task"][:50] for t in memories["traces"]])

## 3. Tools and reasoning memory

With an `Agent` in the pipeline, the writer receives the agent's full transcript and records each tool call in the reasoning trace (failed ones too, with the error). With `assistant="tool_results"`, the companies the tool returned become entities.

In [ ]:
import json

from haystack.components.agents import Agent
from haystack.tools import tool


@tool
def search_companies(country: str, sector: str) -> str:
    """Search a (toy) company database by country and sector. Returns JSON."""
    data = [
        {"name": "Aleph Alpha", "type": "ORGANIZATION", "country": "DE", "sector": "AI"},
        {"name": "DeepL", "type": "ORGANIZATION", "country": "DE", "sector": "AI"},
    ]
    return json.dumps([c for c in data if country.lower() in ("de", "germany")])


agent_pipe = Pipeline()
agent_pipe.add_component("reader", Neo4jMemoryRetriever(store))
agent_pipe.add_component(
    "agent", Agent(chat_generator=OpenAIChatGenerator(model="gpt-5.4-mini"), tools=[search_companies])
)
agent_pipe.add_component(
    "writer", Neo4jMemoryWriter(store, extraction_policy=ExtractionPolicy(user="auto", assistant="tool_results"))
)
agent_pipe.connect("reader.messages", "agent.messages")
agent_pipe.connect("agent.messages", "writer.messages")

task = "Find German AI companies using the company search tool."
session_3 = store.create_session(user_id="pravesh")
out = agent_pipe.run({"reader": {"query": task, "session_id": session_3}, "writer": {"query": task, "session_id": session_3}})
print("Trace:", out["writer"]["trace_id"])
print("Entities from tool results:", [e["name"] for e in out["writer"]["entities"]])

## 4. Save memory directly, without the writer

The store has one method per kind of memory, for importing history, seeding preferences, or recording a task that ran outside a pipeline. They raise on bad input; on NAMS, preferences and facts are skipped with a warning.

In [ ]:
from haystack.dataclasses import ChatMessage, ToolCall

session_4 = store.create_session(user_id="pravesh")
print("Session 4 ID:", session_4)
# Short-term
print(store.add_messages(session_4, [
    ChatMessage.from_user("I mostly invest in seed rounds."),
    ChatMessage.from_assistant("Noted: seed rounds."),
], user_id="pravesh"))

# Long-term
print("preference:", store.add_preference("stage", "Prefers seed-stage companies", user_id="pravesh"))
print("fact:", store.add_fact("Northwind Capital", "headquartered_in", "Berlin"))
print("entity:", store.add_entity("Northwind Capital", "ORGANIZATION", description="Venture capital firm"))

# Reasoning: a task with its tool call and result
call = ToolCall(tool_name="search_companies", arguments={"country": "DE", "sector": "AI"}, id="call-1")
print("trace:", store.add_trace(session_4, "Find seed-stage German AI companies", [
    ChatMessage.from_assistant(tool_calls=[call]),
    ChatMessage.from_tool('[{"name": "Aleph Alpha", "type": "ORGANIZATION"}]', origin=call),
    ChatMessage.from_assistant("Aleph Alpha is the closest match."),
]))

In [ ]:
# Read it back with the retriever
recall = Neo4jMemoryRetriever(store).run(query="seed-stage investing in Germany", session_id=session_4)
print("History:", [m.text for m in recall["history"]])
print(recall["context"])

## 5. Memory as tools: the agent decides when to recall and store

Like Haystack's Mem0 tools: `retrieve_memories` and `store_memory`. `session_id` and `user_id` come from the agent's state, so the model never has to know them.

The model calls `store_memory` only when it decides to, so the system prompt tells it **when** to store and **which memory type** to use, with examples. The tools save notes, not the conversation: `store.add_messages(...)` keeps the chat itself (short-term).

Best with `BACKEND = "bolt"`: NAMS does not store preferences or facts, so on NAMS the tool reports them as not stored.

In [ ]:
from haystack.components.generators.utils import print_streaming_chunk

from haystack_integrations.tools.neo4j import Neo4jMemoryRetrieverTool, Neo4jMemoryWriterTool

MEMORY_SYSTEM_PROMPT = """You are a research assistant for venture investors, with long-term memory.

## Recall
- At the start of every turn, call retrieve_memories WITHOUT a query to see the conversation and the user's preferences.
- Call retrieve_memories WITH a query when you need something specific (a company, a past task).

## Store
Call store_memory whenever the user tells you something that will still matter in a future conversation:
- preference: how they want things. category = short topic.
    "I only want B2B companies"         -> preference, category "business_model", preference "Only B2B companies"
    "Focus on seed rounds"              -> preference, category "stage", preference "Seed-stage companies"
- fact: a stable statement about someone or something, as subject / predicate / object.
    "I work at Northwind Capital"       -> fact: "Pravesh" / "works_at" / "Northwind Capital"
    "We're moving to Munich next month" -> fact: "Pravesh" / "relocating_to" / "Munich"
- entity: an important person, organization or place they care about.
    "Keep an eye on Aleph Alpha"        -> entity: "Aleph Alpha", ORGANIZATION

Rules:
- One store_memory call per item; several in one turn is fine.
- Don't store what retrieve_memories already returned, small talk, or anything only useful for this answer.
- Don't tell the user you stored something unless they ask."""

writer_tool = Neo4jMemoryWriterTool(
    memory_store=store,
    description=(
        "Save one durable piece of information about the user for future conversations: a preference, "
        "a fact (subject, predicate, object) or an entity. Call it as soon as the user states one."
    ),
)

memory_agent = Agent(
    chat_generator=OpenAIChatGenerator(model="gpt-5.4-mini"),
    tools=[Neo4jMemoryRetrieverTool(memory_store=store), writer_tool],
    system_prompt=MEMORY_SYSTEM_PROMPT,
    streaming_callback=print_streaming_chunk,
    state_schema={"session_id": {"type": str}, "user_id": {"type": str}},
)


def show_memory_calls(result):
    """Which memory tools the model called, with what, and what each returned."""
    calls = [(tc.tool_name, tc.arguments) for m in result["messages"] for tc in (m.tool_calls or [])]
    results = [r.result for m in result["messages"] for r in m.tool_call_results]
    for (name, arguments), outcome in zip(calls, results):
        print(f"\n{name}({arguments})\n  -> {outcome}")


session_5 = store.create_session(user_id="pravesh")
result = memory_agent.run(
    messages=[ChatMessage.from_user("I'm moving to Munich next month and I only want B2B companies from now on.")],
    session_id=session_5,
    user_id="pravesh",
)
store.add_messages(session_5, result["messages"], user_id="pravesh")  # also keep the chat (short-term)
show_memory_calls(result)

In [ ]:
# A new session: the agent recalls what it stored
session_6 = store.create_session(user_id="pravesh")
result = memory_agent.run(
    messages=[ChatMessage.from_user("Which companies would suit me?")], session_id=session_6, user_id="pravesh"
)
store.add_messages(session_6, result["messages"], user_id="pravesh")
show_memory_calls(result)

## 6. Look at the graph (bolt) and clean up

In [ ]:
if BACKEND == "bolt":
    rows = store.run(store.client.query.cypher(
        """
        MATCH (p:Preference) RETURN 'preference' AS kind, p.category + ': ' + p.preference AS value
        UNION ALL MATCH (f:Fact) RETURN 'fact' AS kind, f.subject + ' ' + f.predicate + ' ' + f.object AS value
        UNION ALL MATCH (e:Entity) RETURN 'entity' AS kind, e.name + ' (' + e.type + ')' AS value
        """
    ))
    for row in rows:
        print(row)

In [ ]:
# Deleting a session removes its conversation; long-term memory extracted from it is kept by design.
for session in [session_1, session_2, session_3, session_4, session_5, session_6]:
    store.delete_session(session)
store.close()